# 06 – Phân Tích Lát Cắt Thời Gian & Dịch Chuyển Hotspot (M3)

Phụ trách: **M3 – Thành viên 3 (Hoàng Tuấn Bảo)**. Theo đặc tả BA Spec v1.1 (Mục 2.3.5 & 3.4.3):
- **M3-06 Time-slicing**: Phân tích độc lập 3 lát cắt: `WEEKDAY_AM` (07:00–09:59 Mon–Fri), `WEEKDAY_PM` (16:00–18:59 Mon–Fri), và `WEEKEND` (Sat–Sun cả ngày).
- **M3-07 Hotspot Tracking**: Không so sánh trực tiếp nhãn $0/1/2$ (vì định danh ngẫu nhiên giữa các lần fit). Ghép mỗi cụm nguồn với centroid gần nhất trong ngưỡng 3.500 m; khu vực hành chính chỉ được giữ làm metadata diễn giải.
- **Đo lường độ dịch chuyển**: Tính khoảng cách dịch chuyển của tâm cụm $\Delta d$ (mét), sự biến động số chuyến $\Delta n$ và tỷ trọng $\Delta \text{share}$ (%).
- **M3-08 Báo cáo & Bàn giao**: Xuất bảng theo dõi `outputs/metrics/hotspot_tracking.csv`, vẽ biểu đồ `outputs/figures/temporal_hotspot_shifts.png` và bản đồ tương tác vector dịch chuyển `outputs/maps/hdbscan_temporal_shift_map.html` để bàn giao cho M4.

In [ ]:
# ===== Ô THIẾT LẬP (chạy được cả trên Google Colab và máy cá nhân) =====
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/tanbentam/KhaiPhaDuLieu.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = pathlib.Path("/content/KhaiPhaDuLieu")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
    os.chdir(ROOT)
    from src.colab import link_drive_data
    link_drive_data("KhaiPhaDuLieu")
else:
    here = pathlib.Path.cwd().resolve()
    ROOT = next(p for p in (here, *here.parents) if (p / "config" / "experiment.yaml").is_file())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Môi trường Colab:", IN_COLAB, "| Thư mục gốc repo:", ROOT)

## 1. Nạp Dữ Liệu Cụm HDBSCAN Của 3 Lát Cắt

In [ ]:
import json
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.config import load_config, resolve_path
from src.evaluation.temporal import build_time_slice_manifest

cfg = load_config()
sample_id = json.loads(resolve_path(cfg, "master_manifest").read_text(encoding="utf-8"))["sample_id"]
metrics_dir = resolve_path(cfg, "run_log").parent
clp = metrics_dir / f"hdbscan_clusters_{sample_id}.csv"

clusters_df = pd.read_csv(clp)
print(f"Đã nạp {len(clusters_df)} cụm từ {clp.name}")

source_df = pd.read_parquet(resolve_path(cfg, "master"), columns=["time_slice", "trip_start_timestamp"])
slice_manifest = build_time_slice_manifest(source_df, cfg["time_slices"])
slice_manifest["sample_id"] = sample_id
slice_manifest["crs"] = cfg["project"]["crs"]
slice_manifest.to_csv(metrics_dir / "time_slice_manifest.csv", index=False)

metrics = pd.read_csv(metrics_dir / "hdbscan_metrics.csv")
if set(metrics["sample_id"]) != {sample_id}:
    raise ValueError("HDBSCAN metrics do not match the master sample; rerun M3 first.")
count_summary = slice_manifest.merge(
    metrics[["time_slice", "run_id", "n_points", "n_noise", "noise_ratio", "coverage", "n_clusters"]],
    on="time_slice", how="left", validate="one_to_one"
)
if count_summary["run_id"].isna().any() or not np.array_equal(count_summary["n_pickups"], count_summary["n_points"]):
    raise ValueError("Slice counts do not match the HDBSCAN metrics; rerun M3 first.")
for time_slice, slice_clusters in clusters_df.groupby("time_slice"):
    run_id = count_summary.loc[count_summary["time_slice"] == time_slice, "run_id"].iloc[0]
    if set(slice_clusters["run_id"]) != {run_id}:
        raise ValueError(f"Cluster details and metrics have different runs for {time_slice}.")
count_summary["n_clustered"] = count_summary["n_points"] - count_summary["n_noise"]
count_summary["clustered_pickups_per_hour"] = count_summary["n_clustered"] / count_summary["exposure_hours"]
count_summary.to_csv(metrics_dir / "temporal_sample_counts.csv", index=False)
count_summary[["time_slice", "n_days", "exposure_hours", "n_pickups", "pickups_per_hour", "n_noise", "coverage"]]

clusters_df[["time_slice", "cluster", "n_trips", "share", "radius_rms_m", "top_community_area"]].round(3)

## 2. Thuật Toán Ghép Cặp Cụm Giữa Các Lát Cắt (Cluster Matching)

Áp dụng hàm `build_hotspot_tracking` từ `src.evaluation.temporal`:
- Với mỗi cụm ở lát cắt trước, tìm cụm gần nhất ở lát cắt sau với ngưỡng khoảng cách $d \le 3.500\text{ m}$.
- Nếu tìm thấy: đánh dấu `MATCHED`, tính khoảng cách dịch chuyển $\Delta d$, biến động số chuyến và tỷ trọng $\Delta \text{share}$.
- Nếu không tìm thấy: đánh dấu `DISSOLVED` (biến mất) hoặc `EMERGING` (mới xuất hiện).

In [ ]:
from src.evaluation.temporal import build_hotspot_tracking

tracking_df = build_hotspot_tracking(clusters_df, max_distance_m=3500.0)
out_tracking_path = resolve_path(cfg, "run_log").parent / "hotspot_tracking.csv"
tracking_df.to_csv(out_tracking_path, index=False)
print("Đã lưu bảng theo dõi hotspot:", out_tracking_path.relative_to(ROOT))

# Hiển thị các cặp dịch chuyển AM -> PM
show_cols = ["transition", "status", "cluster_from", "cluster_to", "top_area_from", "top_area_to",
             "shift_distance_m", "trips_from", "trips_to", "delta_trips", "delta_share_pct_points"]
tracking_df[tracking_df.transition == "WEEKDAY_AM -> WEEKDAY_PM"][show_cols]

## 3. Trực Quan Hoá Biến Động Nhu Cầu & Tỷ Trọng Hotspot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ 1: Số chuyến đón theo cụm
ax1 = axes[0]
pivot_trips = clusters_df.pivot(index="cluster", columns="time_slice", values="n_trips").fillna(0)
pivot_trips.plot(kind="bar", ax=ax1, width=0.8, colormap="tab10")
ax1.set_title("Quy mô số lượt đón (trips) theo cụm qua các lát cắt", fontsize=11)
ax1.set_xlabel("Cụm (Cluster ID)")
ax1.set_ylabel("Số lượt đón (pickups)")
ax1.grid(axis="y", alpha=0.3)
ax1.legend(title="Lát cắt thời gian")

# Biểu đồ 2: Tỷ trọng (%) của từng cụm
ax2 = axes[1]
pivot_share = (clusters_df.pivot(index="cluster", columns="time_slice", values="share").fillna(0) * 100)
pivot_share.plot(kind="bar", ax=ax2, width=0.8, colormap="tab10")
ax2.set_title("Tỷ trọng nhu cầu (%) của từng cụm qua các lát cắt", fontsize=11)
ax2.set_xlabel("Cụm (Cluster ID)")
ax2.set_ylabel("Tỷ trọng (%)")
ax2.grid(axis="y", alpha=0.3)
ax2.legend(title="Lát cắt thời gian")

fig.suptitle("HDBSCAN: Phân tích biến động Hotspot theo thời gian (Chicago Taxi)", fontsize=13, y=1.02)
fig.tight_layout()
fig_path = resolve_path(cfg, "figures_dir") / "temporal_hotspot_shifts.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print("Đã lưu biểu đồ phân tích thời gian:", fig_path.relative_to(ROOT))

## 4. Bản Đồ Tương Tác Vector Dịch Chuyển Không Gian - Thời Gian

In [ ]:
from src.visualization.maps import temporal_shift_map

map_shift_path = resolve_path(cfg, "maps_dir") / "hdbscan_temporal_shift_map.html"
temporal_shift_map(
    clusters_df,
    tracking_df,
    map_shift_path,
    title="HDBSCAN - Bản đồ dịch chuyển Hotspot Không gian - Thời gian (Chicago)",
    note="Đường nối thể hiện sự dịch chuyển tâm cụm giữa AM -> PM và PM -> Weekend. Dữ liệu master_v1.",
)
print("Đã lưu bản đồ vector dịch chuyển:", map_shift_path.relative_to(ROOT), f"({map_shift_path.stat().st_size / 1024:.0f} KB)")

## 5. Các Phát Hiện Nghiệp Vụ Chính (Business Insights)

1. **Trung tâm thương mại & Hành chính (Loop - Near North Side):**
   - Là điểm nóng lớn nhất ở cả 3 lát cắt (chiếm 60–68% tổng nhu cầu đón xe).
   - Giữa sáng ngày thường (`WEEKDAY_AM`) và chiều ngày thường (`WEEKDAY_PM`), tâm cụm dịch chuyển nhẹ về phía bắc (khu Near North Side và bờ hồ) khi hoạt động chuyển từ văn phòng/công sở sang ăn uống, mua sắm và giải trí ven hồ.
2. **Cụm Sân bay O'Hare (Khu vực 76):**
   - Vị trí tâm cụm cực kỳ ổn định (khoảng cách dịch chuyển $\Delta d < 200\text{ m}$ giữa các lát cắt), ranh giới rất chặt chẽ bao quanh nhà ga sân bay.
   - Quy mô đón xe tại O'Hare tăng mạnh vào buổi chiều ngày thường và duy trì ổn định vào cuối tuần khi khách hạ cánh trở về thành phố.
3. **Cụm Sân bay Midway (Khu vực 56):**
   - Giữ vững quy mô 3–5% nhu cầu xuyến suốt cả tuần.
   - HDBSCAN giữ liền khối cụm Midway thành 1 cụm duy nhất ở cả ngày thường và cuối tuần, khắc phục hoàn toàn hiện tượng DBSCAN làm đứt Midway thành 2 mảnh vụn vào cuối tuần.
4. **Cụm Giáo dục & Nghiên cứu Hyde Park (Khu vực 41 - ĐH Chicago):**
   - Hoạt động rõ rệt nhất vào sáng ngày thường (`WEEKDAY_AM`), sau đó phân tán vào buổi chiều và cuối tuần.